In [1]:
import pandas as pd
import sys
from pathlib import Path
import duckdb
import polars as pl
from deltalake import DeltaTable
sys.path.append(str(Path("..").resolve()))
from src.services import bloomberg_client

In [ ]:
mypath = r'C:/Users/mmoin/PYTHON PROJECTS/DataWareHouse/bronze/harvest_fund_identifiers'
con = duckdb.connect()
con.execute("INSTALL DELTA;")
con.execute("LOAD DELTA;")
ans = con.execute(f"select ticker from delta_scan('{mypath}') where ticker is not null").pl()
ticker_list = ans.with_columns(pl.col('ticker')+' CN Equity').to_series().to_list()
print(ticker_list)

In [ ]:
mypath2 = r'C:/Users/mmoin/PYTHON PROJECTS/DataWareHouse/bronze/history_all_distributions'
con = duckdb.connect()
con.execute("INSTALL DELTA;")
con.execute("LOAD DELTA;")
ans = con.execute(f"""select distinct "Ex-Date" from delta_scan('{mypath2}')""").pl()
dates = ans.with_columns(pl.col('Ex-Date')).to_series().to_list()
dates = [date.replace("-", "") for date in dates]
print(dates)

In [ ]:

with bloomberg_client.BloombergClient() as client:
    mydf = pd.DataFrame()
    for fund in ticker_list:
        ref = client.BDS(fund, "DVD_HIST_ALL")
        
        # Only convert columns that exist in the DataFrame
        dtype_map = {
            "Ex-Date": "datetime64[ns]",
            "Record Date": "datetime64[ns]",
            "Payable Date": "datetime64[ns]",
            "Declared Date": "datetime64[ns]",
            "Dividend Amount": float
        }
        
        # Filter to only columns that exist
        dtype_map = {col: dtype for col, dtype in dtype_map.items() if col in ref.columns}
        ref = ref.astype(dtype_map)
        ref['ticker'] = fund  # Add ticker column to identify the fund
        
        mydf = pd.concat([mydf, ref], ignore_index=True)


In [ ]:
with bloomberg_client.BloombergClient() as client:
    ans = pd.DataFrame()
    for mydate in dates:
        ref = client.BDH(["HBFE CN Equity","HLFE CN Equity"], ["FUND_NET_ASSET_VAL"], mydate , mydate)
        ans = pd.concat([ans, ref]).reset_index(drop=True)
print(ans.head())

In [ ]:
mypath = Path(r'C:\Users\mmoin\PYTHON PROJECTS\DataWareHouse\silver\ref_all_securities')
# Use DeltaTable instead of pl.scan_delta() due to compatibility issues
delta_table = DeltaTable(str(mypath))
mysecs = delta_table.to_pandas()['figi_id'].tolist()
print(f"Loaded {len(mysecs)} securities")

with bloomberg_client.BloombergClient() as client:
    ref = client.BDP(mysecs, ["COMPOSITE_EXCH_CODE", "CNTRY_ISSUE_ISO"])
print(ref.head())

Loaded 878 securities
         Ticker COMPOSITE_EXCH_CODE
0  BBG020PC6P92                None
1  BBG02160LKJ0                None
2  BBG000BPD168                  US
3  BBG01WQ51HY2                None
4  BBG000LYF3S8                  US


In [5]:
ref["COMPOSITE_EXCH_CODE"].unique()

array([None, 'US', 'SM', 'DC', 'FP', 'FH', 'NA', 'LN', 'SW', 'GR', 'NZ',
       'CN', 'AU', 'PL', 'IM', 'HK', 'SP', 'NO', 'SS', 'AV'], dtype=object)